# Camera Calibration and Tap-Drive Validation

This notebook is the working calibration notebook for the front camera and tap-drive feature.

It is intentionally safe to run before real calibration images are added. In that state it reports which inputs are missing and displays the available CSV templates instead of producing false calibration results.


In [1]:
from pathlib import Path
import json

import pandas as pd
import matplotlib.pyplot as plt

cwd = Path.cwd().resolve()
if (cwd / "tests" / "results").exists():
    ROOT = cwd
elif cwd.name.lower() == "tests":
    ROOT = cwd.parent
elif cwd.parent.name.lower() == "tests":
    ROOT = cwd.parent.parent
else:
    raise FileNotFoundError("Run from repository root, tests, or tests/notebooks.")

RESULTS = ROOT / "tests" / "results"
CALIBRATION_IMAGES = ROOT / "tests" / "camera_calibration_images" / "front"
OUTPUT_INTRINSICS = RESULTS / "camera_intrinsics_front.json"
OUTPUT_HOMOGRAPHY = RESULTS / "camera_homography_front.json"
HOMOGRAPHY_TEMPLATE = RESULTS / "camera_homography_points_template.csv"
TAP_TEMPLATE = RESULTS / "tap_drive_accuracy_test_template.csv"
TAP_RESULTS = RESULTS / "tap_drive_accuracy_test.csv"

print(f"Repository root: {ROOT}")
print(f"Calibration images folder: {CALIBRATION_IMAGES}")


Repository root: C:\Users\jevgeni.gandsu\OneDrive - FINESTMEDIA AS\Desktop\UT\Prototyping\repo
Calibration images folder: C:\Users\jevgeni.gandsu\OneDrive - FINESTMEDIA AS\Desktop\UT\Prototyping\repo\tests\camera_calibration_images\front


## 1. Input Inventory

Add front-camera chessboard images under `tests/camera_calibration_images/front/` before running intrinsic calibration.


In [2]:
image_files = []
if CALIBRATION_IMAGES.exists():
    image_files = sorted([p for p in CALIBRATION_IMAGES.iterdir() if p.suffix.lower() in {'.png', '.jpg', '.jpeg'}])

inventory = pd.DataFrame({
    "input": ["calibration_images", "homography_points", "tap_drive_results"],
    "path": [str(CALIBRATION_IMAGES), str(HOMOGRAPHY_TEMPLATE), str(TAP_RESULTS)],
    "available": [bool(image_files), HOMOGRAPHY_TEMPLATE.exists(), TAP_RESULTS.exists()],
    "count_or_note": [len(image_files), "template exists" if HOMOGRAPHY_TEMPLATE.exists() else "missing", "measured CSV exists" if TAP_RESULTS.exists() else "not measured yet"],
})
inventory


,input,path,available,count_or_note
0,calibration_images,C:\Users\jevgeni.gandsu\OneDrive - FINESTMEDIA...,False,0
1,homography_points,C:\Users\jevgeni.gandsu\OneDrive - FINESTMEDIA...,True,template exists
2,tap_drive_results,C:\Users\jevgeni.gandsu\OneDrive - FINESTMEDIA...,False,not measured yet


## 2. Homography Point Template

Fill this table with measured arena marker coordinates and corresponding pixel positions from the camera image.


In [3]:
homography_points = pd.read_csv(HOMOGRAPHY_TEMPLATE) if HOMOGRAPHY_TEMPLATE.exists() else pd.DataFrame()
homography_points


,point_id,pixel_x,pixel_y,arena_x_mm,arena_y_mm,notes
0,P1,NaN,NaN,NaN,NaN,NaN
1,P2,NaN,NaN,NaN,NaN,NaN
2,P3,NaN,NaN,NaN,NaN,NaN
3,P4,NaN,NaN,NaN,NaN,NaN
4,P5,NaN,NaN,NaN,NaN,NaN
5,P6,NaN,NaN,NaN,NaN,NaN


In [4]:
numeric_cols = ["pixel_x", "pixel_y", "arena_x_mm", "arena_y_mm"]
if not homography_points.empty:
    usable = homography_points.dropna(subset=numeric_cols)
    print(f"Usable homography points: {len(usable)}")
    if len(usable) >= 4:
        fig, ax = plt.subplots(1, 2, figsize=(10, 4))
        ax[0].scatter(usable["pixel_x"], usable["pixel_y"])
        ax[0].invert_yaxis()
        ax[0].set_title("Image marker points")
        ax[0].set_xlabel("pixel x")
        ax[0].set_ylabel("pixel y")
        ax[1].scatter(usable["arena_x_mm"], usable["arena_y_mm"])
        ax[1].set_aspect("equal", adjustable="box")
        ax[1].set_title("Arena marker points")
        ax[1].set_xlabel("x, mm")
        ax[1].set_ylabel("y, mm")
        plt.tight_layout()
        plt.show()
    else:
        print("Need at least 4 filled marker rows before estimating homography.")


Usable homography points: 0
Need at least 4 filled marker rows before estimating homography.


## 3. Tap-Drive Accuracy Template

Use the template below for measured tap-drive target trials. Do not rename the filled file to `tap_drive_accuracy_test.csv` until it contains real measured rows.


In [5]:
tap_path = TAP_RESULTS if TAP_RESULTS.exists() else TAP_TEMPLATE
tap = pd.read_csv(tap_path) if tap_path.exists() else pd.DataFrame()
print(f"Loaded: {tap_path.name if tap_path.exists() else 'missing'}")
tap


Loaded: tap_drive_accuracy_test_template.csv


,date,target_id,tap_x_px,tap_y_px,target_x_mm,target_y_mm,final_x_mm,final_y_mm,position_error_mm,heading_error_deg,result,notes
0,YYYY-MM-DD,T1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,YYYY-MM-DD,T2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,YYYY-MM-DD,T3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,YYYY-MM-DD,T4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,YYYY-MM-DD,T5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
if TAP_RESULTS.exists():
    measured = tap.dropna(subset=["position_error_mm"])
    if not measured.empty:
        summary = measured.agg(
            trials=("target_id", "count"),
            median_error_mm=("position_error_mm", "median"),
            max_error_mm=("position_error_mm", "max"),
            median_heading_error_deg=("heading_error_deg", "median"),
        )
        display(summary)
        fig, ax = plt.subplots()
        ax.bar(measured["target_id"].astype(str), measured["position_error_mm"])
        ax.set_title("Tap-drive final position error")
        ax.set_xlabel("Target")
        ax.set_ylabel("Error, mm")
        plt.tight_layout()
        plt.show()
    else:
        print("tap_drive_accuracy_test.csv exists, but contains no measured position_error_mm values.")
else:
    print("No measured tap-drive result CSV yet. Fill the template after physical testing.")


No measured tap-drive result CSV yet. Fill the template after physical testing.


## 4. Calibration Status

Complete status requires real chessboard images, generated intrinsics JSON, generated homography JSON, and measured tap-drive rows. Until then this notebook is evidence of the planned calibration workflow, not evidence of completed calibration.
